In [7]:
"""
CISB5123 Text Analytics - Lab Assignment 2
Name: Faez Nazran Haziq Bin Yusrizal, Muhammad Harizul Zharfan Bin Abdul Karim
Student ID: IS01083871, IS01083880

Discussion on Strengths and Weaknesses:
The lexicon-based model is very easy to understand and build because it just counts positive and 
negative words. However, its main weakness is that it isn't very smart; it doesn't understand context 
or sarcasm, and if a word isn't in our list, it ignores it. 
On the other hand, Machine Learning models (like Naive Bayes and Logistic Regression) are much stronger. 
Their strength is that they actually learn patterns from the dataset itself, which gives them much higher 
accuracy. TF-IDF is especially good because it lowers the value of words that appear too often. The only 
weakness of machine learning is that it takes more time to process and requires the data to be split into 
training and testing sets first.
"""

import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# STEP 1: LOAD AND PREPARE DATA
print("Loading data...")
df = pd.read_csv("Reviews.csv")

# Drop any rows that have missing text or scores
df = df.dropna(subset=["Text", "Score"])

# Take a sample of 10,000 rows so our computer doesn't freeze processing 500k rows
df = df.sample(n=10000, random_state=42)

# Create a function to change 1-5 scores into words
def make_sentiment_label(score):
    if score >= 4:
        return "Positive"
    elif score == 3:
        return "Neutral"
    else:
        return "Negative"

# Apply the function to create a new column
df["Sentiment"] = df["Score"].apply(make_sentiment_label)


# TEXT CLEANING
def simple_clean_text(text):
    # 1. Make everything lowercase
    text = str(text).lower()
    # 2. Remove HTML tags like <br>
    text = re.sub(r'<.*?>', ' ', text)
    # 3. Remove punctuation and numbers (keep only a-z and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

print("Cleaning the text...")
df["Clean_Text"] = df["Text"].apply(simple_clean_text)

# Save the cleaned data to a CSV for the assignment deliverable
df[["Score", "Sentiment", "Text", "Clean_Text"]].to_csv("processed_review_updated.csv", index=False)
print("Saved cleaned data to processed_review_updated.csv\n")


# SPLIT THE DATA FOR MACHINE LEARNING
X_train, X_test, y_train, y_test = train_test_split(
    df["Clean_Text"], 
    df["Sentiment"], 
    test_size=0.2, 
    random_state=42
)


# LEXICON-BASED MODEL (Simple Word Counting)
print("--- MODEL 1: LEXICON BASED ---")

my_positive_words = ["good", "great", "love", "best", "delicious", "perfect", "nice", "excellent"]
my_negative_words = ["bad", "terrible", "awful", "worst", "hate", "gross", "disgusting", "poor"]

def lexicon_classifier(text):
    words = text.split()
    pos_score = 0
    neg_score = 0
    
    for word in words:
        if word in my_positive_words:
            pos_score += 1
        elif word in my_negative_words:
            neg_score += 1
            
    if pos_score > neg_score:
        return "Positive"
    elif neg_score > pos_score:
        return "Negative"
    else:
        return "Neutral"

lexicon_predictions = X_test.apply(lexicon_classifier)

lexicon_acc = accuracy_score(y_test, lexicon_predictions)
print(f"Lexicon Accuracy: {lexicon_acc:.2f}")
print("Lexicon Report:")
print(classification_report(y_test, lexicon_predictions))


# MACHINE LEARNING - BAG OF WORDS + NAIVE BAYES
print("--- MODEL 2: BAG OF WORDS + NAIVE BAYES ---")

# Feature Extraction: Convert words to counts
bow_vectorizer = CountVectorizer(stop_words='english')
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Train the model
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

# Test the model
nb_predictions = nb_model.predict(X_test_bow)

nb_acc = accuracy_score(y_test, nb_predictions)
print(f"Bag of Words + Naive Bayes Accuracy: {nb_acc:.2f}")
print(classification_report(y_test, nb_predictions))


# MACHINE LEARNING - TF-IDF + LOGISTIC REGRESSION
print("--- MODEL 3: TF-IDF + LOGISTIC REGRESSION ---")

# Feature Extraction: Convert words to TF-IDF scores
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Train the model
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# Test the model
lr_predictions = lr_model.predict(X_test_tfidf)

lr_acc = accuracy_score(y_test, lr_predictions)
print(f"TF-IDF + Logistic Regression Accuracy: {lr_acc:.2f}")
print(classification_report(y_test, lr_predictions))


# FINAL COMPARISON
print("--- FINAL RESULTS SUMMARY ---")
print(f"1. Lexicon Method Accuracy:      {lexicon_acc:.2f}")
print(f"2. Naive Bayes (BoW) Accuracy:   {nb_acc:.2f}")
print(f"3. Logistic Reg (TF-IDF) Accuracy: {lr_acc:.2f}")

Loading data...
Cleaning the text...
Saved cleaned data to processed_review_updated.csv

--- MODEL 1: LEXICON BASED ---
Lexicon Accuracy: 0.62
Lexicon Report:
              precision    recall  f1-score   support

    Negative       0.53      0.14      0.22       281
     Neutral       0.10      0.42      0.16       144
    Positive       0.87      0.73      0.79      1575

    accuracy                           0.62      2000
   macro avg       0.50      0.43      0.39      2000
weighted avg       0.77      0.62      0.67      2000

--- MODEL 2: BAG OF WORDS + NAIVE BAYES ---
Bag of Words + Naive Bayes Accuracy: 0.83
              precision    recall  f1-score   support

    Negative       0.79      0.35      0.49       281
     Neutral       0.14      0.01      0.01       144
    Positive       0.83      0.99      0.90      1575

    accuracy                           0.83      2000
   macro avg       0.59      0.45      0.47      2000
weighted avg       0.78      0.83      0.78     